SMART WASTE COLLECTION SIMULATION

Data Understanding by Singaravel Durga Shankari - G2607549H

Checking if the file is loaded successfully

In [ ]:
import json

with open("wyndham_smartbin_filllevel.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print("File loaded successfully!")
print("Number of records:", len(data["features"]))

1 — Understanding the top-level JSON structure

In [ ]:
print("\nTop-level keys:")
print(data.keys())
print("\nFirst feature:")
print(data["features"][0])

2 — Looking at more records

In [ ]:
print("\nFirst 5 records:")

for i in range(5):
    print(f"\nRecord {i + 1}")
    print(data["features"][i]["properties"])

3 — Converting the JSON into a table

In [ ]:
import pandas as pd

records = []
for feature in data["features"]:
    properties = feature["properties"]
    coordinates = feature["geometry"]["coordinates"]
    record = properties.copy()
    record["longitude"] = coordinates[0]
    record["latitude"] = coordinates[1]
    records.append(record)

df = pd.DataFrame(records)

print("\nDataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

4 — Checking data types

In [ ]:
print("\nData types:")
print(df.dtypes)

5 — Checking how many actual smart bins are there?

In [ ]:
print("\nNumber of unique smart bins:")
print(df["serialNumber"].nunique())
print("\nUnique bin serial numbers:")
print(df["serialNumber"].unique())

6 — Checking how many observations do we have for each bin?

In [ ]:
observations_per_bin = df["serialNumber"].value_counts().sort_index()

print("\nNumber of observations per bin:")
print(observations_per_bin)
print("\nObservation count summary:")
print(observations_per_bin.describe())

7 — Checking the time available in the data

In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])

print("\nTimestamp data type:")
print(df["timestamp"].dtype)

print("\nEarliest date:")
print(df["timestamp"].min())

print("\nLatest date:")
print(df["timestamp"].max())

print("\nNumber of unique dates:")
print(df["timestamp"].nunique())

8 — Checking how many bins are recorded on each date?

In [ ]:
bins_per_date = df.groupby("timestamp")["serialNumber"].nunique()

print("\nNumber of bins recorded per date:")
print(bins_per_date.describe())
print("\nFirst 10 dates:")
print(bins_per_date.head(10))

print("\nLast 10 dates:")
print(bins_per_date.tail(10))

9 — Understanding the fullness scale of the bins

In [ ]:
print("\nUnique fullness values:")
print(sorted(df["latestFullness"].unique()))

print("\nNumber of unique fullness values:")
print(df["latestFullness"].nunique())

print("\nFrequency of each fullness value:")
print(df["latestFullness"].value_counts().sort_index())

print("\nFullness summary:")
print(df["latestFullness"].describe())

10 — What is in the reason column

In [ ]:
print("\nUnique reason values:")
print(df["reason"].unique())

print("\nNumber of unique reason values:")
print(df["reason"].nunique())

print("\nFrequency of each reason:")
print(df["reason"].value_counts(dropna=False))

print("\nFullness by reason:")
print(pd.crosstab(df["reason"], df["latestFullness"]))

11 — What is in the fullnessThreshold

In [ ]:
print("\nUnique fullness threshold values:")
print(sorted(df["fullnessThreshold"].unique()))

print("\nFrequency of fullness thresholds:")
print(df["fullnessThreshold"].value_counts().sort_index())

print("\nReason by fullness threshold:")
print(pd.crosstab(df["fullnessThreshold"], df["reason"]))

print("\nFullness level by fullness threshold:")
print(pd.crosstab(df["fullnessThreshold"], df["latestFullness"]))

12 — What is the threshold that each bin uses

In [ ]:
print("\nThresholds used by each bin:")

threshold_by_bin = (
    df.groupby("serialNumber")["fullnessThreshold"]
      .unique()
)

print(threshold_by_bin)

print("\nNumber of bins by threshold:")

print(
    df.groupby("fullnessThreshold")["serialNumber"]
      .nunique()
)

13 — Checking ageThreshold and position

In [ ]:
print("\nUnique ageThreshold values:")
print(df["ageThreshold"].unique())

print("\nAge threshold frequency:")
print(df["ageThreshold"].value_counts(dropna=False))

print("\nUnique position values:")
print(df["position"].unique())

print("\nPosition frequency:")
print(df["position"].value_counts(dropna=False))

14 — What are the 33 smart bins?

In [ ]:
bin_summary = (
    df.groupby("serialNumber")
      .agg(
          description=("description", "first"),
          longitude=("longitude", "first"),
          latitude=("latitude", "first"),
          fullnessThreshold=("fullnessThreshold", "first"),
          observations=("serialNumber", "size")
      )
      .reset_index()
)

print(bin_summary.to_string(index=False))

15 — Understanding the  bin groups

In [ ]:
def get_area(description):
    if "Werribee" in description:
        return "Werribee"
    elif "Point Cook" in description:
        return "Point Cook"
    else:
        return "Other"

df["area"] = df["description"].apply(get_area)

print("\nNumber of unique bins by area:")
print(df.groupby("area")["serialNumber"].nunique())

def get_bin_code(description):
    words = description.split()

    if "G" in words:
        return "G"
    elif "R" in words:
        return "R"
    else:
        return "Unknown"

df["binCode"] = df["description"].apply(get_bin_code)

print("\nNumber of unique bins by apparent bin code:")
print(df.groupby("binCode")["serialNumber"].nunique())

print("\nBins by area and apparent bin code:")
print(
    df.groupby(["area", "binCode"])["serialNumber"]
      .nunique()
)

16 — observe a bin filling up and then being emptied

using 1511190 — Aldi G Point Cook

In [ ]:
bin_1511190 = (
    df[df["serialNumber"] == 1511190]
    .sort_values("timestamp")
)

print(
    bin_1511190[
        ["timestamp", "latestFullness", "reason"]
    ].head(40).to_string(index=False)
)

17 — Checking whether the same behaviour exists across other bins

In [ ]:
sample_bins = [1511191, 1511211]

for bin_id in sample_bins:

    sample = (
        df[df["serialNumber"] == bin_id]
        .sort_values("timestamp")
    )

    print("\n" + "=" * 60)
    print("BIN:", bin_id)
    print("DESCRIPTION:", sample["description"].iloc[0])
    print("=" * 60)

    print(
        sample[
            ["timestamp", "latestFullness", "reason"]
        ].head(30).to_string(index=False)
    )

18 — Are locations fixed for each bin?

In [ ]:
location_counts = (
    df.groupby("serialNumber")
      .agg(
          unique_longitudes=("longitude", "nunique"),
          unique_latitudes=("latitude", "nunique")
      )
)

print(location_counts)

print("\nMaximum number of longitude values for one bin:")
print(location_counts["unique_longitudes"].max())

print("\nMaximum number of latitude values for one bin:")
print(location_counts["unique_latitudes"].max())